In [2]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=5622afcf2226cfae394f77a657a2920cfedce1a293488aba8f7c64bc9967a0d7
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [3]:
from pyspark.sql import SparkSession
session= SparkSession.builder.appName("sparkstreaming").getOrCreate()

In [5]:
#Create a folder named AttritionDir in the files section

In [4]:
#Reading the data
empdata=session.read.csv('EmployeeAttrition.csv', header=True, inferSchema=True)

#Reading the stream of data by specifying the schema and directory
empdata_stream=session.readStream.schema(empdata.schema).csv('AttritionDir/')

#Writing the stream of data to a table named emptable
#The output mode is append which means that the data will be added always to existing table
empquery=empdata_stream.writeStream.queryName("employeetable").format("memory").outputMode("append").start()


In [6]:
#Copy the EmployeeAttrition.csv file to the Attrition folder
import shutil
src=r"EmployeeAttrition.csv"
dest = r"AttritionDir"
shutil.copy(src,dest)


'AttritionDir/EmployeeAttrition.csv'

In [7]:
session.sql("select Department, count(*) from employeetable group by Department").show()

+--------------------+--------+
|          Department|count(1)|
+--------------------+--------+
|               Sales|     446|
|          Department|       1|
|Research & Develo...|     961|
|     Human Resources|      63|
+--------------------+--------+



In [8]:
#Displaying the number of records after adding the copy of file to the folder
import time
for i in range (12):
    session.sql("select Department, count(*) from employeetable group by Department").show()
    newfile="AttritionDir/Emp" + str(i) +  ".csv"
    shutil.copy(src,newfile)
    time.sleep(5)

+--------------------+--------+
|          Department|count(1)|
+--------------------+--------+
|               Sales|     446|
|          Department|       1|
|Research & Develo...|     961|
|     Human Resources|      63|
+--------------------+--------+

+--------------------+--------+
|          Department|count(1)|
+--------------------+--------+
|               Sales|     892|
|          Department|       2|
|Research & Develo...|    1922|
|     Human Resources|     126|
+--------------------+--------+

+--------------------+--------+
|          Department|count(1)|
+--------------------+--------+
|               Sales|    1338|
|          Department|       3|
|Research & Develo...|    2883|
|     Human Resources|     189|
+--------------------+--------+

+--------------------+--------+
|          Department|count(1)|
+--------------------+--------+
|               Sales|    1784|
|          Department|       4|
|Research & Develo...|    3844|
|     Human Resources|     252|
+----

In [9]:
#Practical Exercise: Display the total daily rate paid to employees belonging to different marital status

In [10]:
empquery.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [11]:
empquery.stop()

In [12]:
empquery.status

{'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}